In [4]:
import math
import torch
import torch.nn as nn
import math
import torch
import torch.nn as nn
import torch
from transformers import AutoTokenizer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

tokenizer_name = 'Helsinki-NLP/opus-mt-en-vi'
en_tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
vi_tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # Shape: (1, max_len, d_model)
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x: (Batch_size, Seq_len, d_model)
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

class Seq2SeqTransformerOptimized(nn.Module):
    def __init__(
        self,
        src_vocab_size,
        tgt_vocab_size,
        d_model=512,
        nhead=8,
        num_encoder_layers=4,
        num_decoder_layers=4,
        dim_feedforward=1024,
        dropout=0.1,
        pad_idx=53684
    ):
        super().__init__()
        self.d_model = d_model
        self.pad_idx = pad_idx

        self.src_embedding = nn.Embedding(src_vocab_size, d_model)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model, dropout=dropout)

        # 1. BẬT norm_first=True (Pre-LayerNorm) ĐỂ ỔN ĐỊNH GRADIENT
        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
            norm_first=True
        )

        self.fc_out = nn.Linear(d_model, tgt_vocab_size, bias=False)

        # 2. WEIGHT TYING (Chia sẻ ma trận trọng số)
        self.fc_out.weight = self.tgt_embedding.weight

    def create_mask(self, src, tgt):
        tgt_len = tgt.size(1)
        device = src.device

        # Đưa tgt_mask về boolean (dtype=torch.bool) để khớp hoàn toàn với key_padding_mask
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt_len, device=device).bool()

        src_key_padding_mask = (src == self.pad_idx)
        tgt_key_padding_mask = (tgt == self.pad_idx)

        return tgt_mask, src_key_padding_mask, tgt_key_padding_mask

    def forward(self, src, tgt):
        tgt_mask, src_padding_mask, tgt_padding_mask = self.create_mask(src, tgt)

        src_emb = self.pos_encoder(self.src_embedding(src) * math.sqrt(self.d_model))
        tgt_emb = self.pos_encoder(self.tgt_embedding(tgt) * math.sqrt(self.d_model))

        out = self.transformer(
            src=src_emb,
            tgt=tgt_emb,
            tgt_mask=tgt_mask,
            src_key_padding_mask=src_padding_mask,
            tgt_key_padding_mask=tgt_padding_mask,
            memory_key_padding_mask=src_padding_mask
        )
        return self.fc_out(out)

    def encode(self, src):
        src_emb = self.pos_encoder(self.src_embedding(src) * math.sqrt(self.d_model))
        src_padding_mask = (src == self.pad_idx)
        return self.transformer.encoder(src_emb, src_key_padding_mask=src_padding_mask)

    def decode(self, tgt, memory, src_padding_mask=None):
        tgt_len = tgt.size(1)
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt_len, device=tgt.device)
        tgt_emb = self.pos_encoder(self.tgt_embedding(tgt) * math.sqrt(self.d_model))

        out = self.transformer.decoder(
            tgt=tgt_emb,
            memory=memory,
            tgt_mask=tgt_mask,
            memory_key_padding_mask=src_padding_mask
        )
        return (out)

model = Seq2SeqTransformerOptimized(
    src_vocab_size=len(en_tokenizer),
    tgt_vocab_size=len(vi_tokenizer),
    d_model=512,
    nhead=8,
    num_encoder_layers=4,
    num_decoder_layers=4,
    dim_feedforward=1024,
    dropout=0.1,
    pad_idx=vi_tokenizer.pad_token_id
).to(device)



import torch
import torch.nn as nn
import math

def get_banned_tokens(tokens, ngram_size):
    """
    Tìm danh sách token nếu sinh ra sẽ tạo thành n-gram bị trùng lặp.
    """
    if len(tokens) < ngram_size - 1:
        return []
    
    # Lấy tiền tố gồm (n - 1) token gần nhất
    prefix = tuple(tokens[-(ngram_size - 1):])
    banned = []
    
    # Quét trong quá khứ xem prefix này đã từng đi liền với token nào chưa
    for i in range(len(tokens) - ngram_size + 1):
        if tuple(tokens[i : i + ngram_size - 1]) == prefix:
            banned.append(tokens[i + ngram_size - 1])
            
    return banned

checkpoint = torch.load("/kaggle/input/notebooks/khnhhngnguynhu/train-phomt-mt/checkpoints/bestv4.pt"
                        , map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
@torch.no_grad()
@torch.no_grad()
def inference_beam_search(
    source_text: str,
    model: nn.Module,
    source_tokenizer,
    target_tokenizer,
    device: torch.device,
    beam_size: int = 4,
    max_len: int = 60,
    no_repeat_ngram_size: int = 3,
    length_penalty_alpha: float = 0.6
) -> str:
    model.eval()

    # 1. Tokenize câu nguồn
    source_inputs = source_tokenizer(
        source_text,
        return_tensors="pt",
        truncation=True,
        max_length=max_len
    )
    src = source_inputs["input_ids"].to(device)
    src_padding_mask = (src == model.pad_idx)

    # 2. Encode 1 lần duy nhất qua Encoder
    memory = model.encode(src)

    # 3. SỬA TẠI ĐÂY: MarianMT bắt buộc dùng pad_token_id (53684) làm token mồi BOS
    bos_id = target_tokenizer.pad_token_id
    eos_id = target_tokenizer.eos_token_id if target_tokenizer.eos_token_id is not None else 0

    candidates = [([bos_id], 0.0)]
    completed_hypotheses = []

    # 4. Vòng lặp giải mã Beam Search
    for step in range(max_len):
        all_candidates = []

        tgt_tensor = torch.tensor([c[0] for c in candidates], device=device)
        cur_beam_size = tgt_tensor.size(0)

        memory_expanded = memory.repeat(cur_beam_size, 1, 1)
        mask_expanded = src_padding_mask.repeat(cur_beam_size, 1)

        # dec_output trả về hidden state (cur_beam_size, cur_len, 512)
        dec_output = model.decode(tgt_tensor, memory_expanded, src_padding_mask=mask_expanded)
        
        # fc_out chiếu từ 512 -> 53685
        logits = model.fc_out(dec_output[:, -1, :])
        log_probs = torch.log_softmax(logits, dim=-1)

        # 5. Phạt n-gram lặp lại và mở rộng các beam
        for beam_idx, (tokens, cum_score) in enumerate(candidates):
            banned_tokens = get_banned_tokens(tokens, no_repeat_ngram_size)
            if banned_tokens:
                log_probs[beam_idx, banned_tokens] = -float("inf")

            topk_log_probs, topk_tokens = torch.topk(log_probs[beam_idx], k=beam_size)

            for lp, tok in zip(topk_log_probs.tolist(), topk_tokens.tolist()):
                new_tokens = tokens + [tok]
                new_score = cum_score + lp
                all_candidates.append((new_tokens, new_score))

        # 6. Sắp xếp và chọn top beam_size
        all_candidates = sorted(all_candidates, key=lambda x: x[1], reverse=True)
        
        candidates = []
        for tokens, score in all_candidates:
            if tokens[-1] == eos_id:
                seq_len = len(tokens) - 1
                length_penalty = ((5.0 + seq_len) / 6.0) ** length_penalty_alpha
                completed_hypotheses.append((tokens, score / length_penalty))
            else:
                candidates.append((tokens, score))

            if len(candidates) == beam_size:
                break

        if len(completed_hypotheses) >= beam_size or len(candidates) == 0:
            break

    # 7. Lựa chọn câu tốt nhất
    if completed_hypotheses:
        best_tokens, _ = max(completed_hypotheses, key=lambda x: x[1])
    else:
        best_tokens, _ = max(
            candidates, 
            key=lambda x: x[1] / (((5.0 + len(x[0]) - 1) / 6.0) ** length_penalty_alpha)
        )

    # 8. Decode kết quả (bỏ token BOS ban đầu)
    return target_tokenizer.decode(best_tokens[1:], skip_special_tokens=True)

/usr/local/lib/python3.13/dist-packages/torch/nn/modules/transformer.py:144: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = TransformerEncoder(


In [10]:
test_sentences = [
    "Hello, how are you?",
    "Artificial intelligence is developing very fast.",
    "The students are studying hard for the final exam.",
    "I love you.",
    "Would you like to drink something?"
]

for en_text in test_sentences:
    vi_text_beam = inference_beam_search(
        source_text=en_text,
        model=model,
        source_tokenizer=en_tokenizer,
        target_tokenizer=vi_tokenizer,
        device=device,
        beam_size=4,
        max_len=50,
        no_repeat_ngram_size=3
    )
    print(f"EN:   {en_text}")
    print(f"Beam: {vi_text_beam}")
    print("-" * 50)

EN:   Hello, how are you?
Beam: anh thế nào rồi?
--------------------------------------------------
EN:   Artificial intelligence is developing very fast.
Beam: trí tuệ nhân tạo đang phát triển rất nhanh.
--------------------------------------------------
EN:   The students are studying hard for the final exam.
Beam: học sinh đang học chăm chỉ cho kỳ thi cuối cùng.
--------------------------------------------------
EN:   I love you.
Beam: em yêu anh.
--------------------------------------------------
EN:   Would you like to drink something?
Beam: uống gì không?
--------------------------------------------------


In [2]:
!pip install -q sacrebleu evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 6.9 MB/s eta 0:00:00


In [7]:
import os
import math
import torch
import torch.nn as nn
from tqdm import tqdm
from datasets import load_from_disk
from transformers import AutoTokenizer
import sacrebleu

# 1. Cấu hình thiết bị & Tokenizer
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer_name = 'Helsinki-NLP/opus-mt-en-vi'
tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)

# 2. Đường dẫn dữ liệu & Checkpoint
SAVE_DATA = "/kaggle/input/datasets/khnhhngnguynhu/dataset-phomt/datasets"
CHECKPOINT_PATH = "/kaggle/input/notebooks/khnhhngnguynhu/train-phomt-mt/checkpoints/bestv4.pt"

# Nạp split test
loaded_datasets = load_from_disk(SAVE_DATA)
test_dataset = loaded_datasets['test']
print(f"So luong mau tap Test: {len(test_dataset)}")

# 3. Khởi tạo kiến trúc và nạp trọng số mô hình
model = Seq2SeqTransformerOptimized(
    src_vocab_size=len(tokenizer),
    tgt_vocab_size=len(tokenizer),
    d_model=512,
    nhead=8,
    num_encoder_layers=4,
    num_decoder_layers=4,
    dim_feedforward=1024,
    dropout=0.1,
    pad_idx=tokenizer.pad_token_id
).to(device)

print(f"Dang nap checkpoint tu: {CHECKPOINT_PATH}")
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print(f"Nap thanh cong! Epoch: {checkpoint.get('epoch')} | Best Val Loss: {checkpoint.get('val_loss', 0.0):.4f}")

# 4. Hàm Beam Search tối ưu hóa cho tập Test
@torch.no_grad()
def translate_sentence(
    source_text: str,
    beam_size: int = 4,
    max_len: int = 60,
    no_repeat_ngram_size: int = 3,
    length_penalty_alpha: float = 0.6
) -> str:
    # Mã hóa câu nguồn
    source_inputs = tokenizer(
        source_text,
        return_tensors="pt",
        truncation=True,
        max_length=max_len
    )
    src = source_inputs["input_ids"].to(device)
    src_padding_mask = (src == model.pad_idx)

    # Encode qua Encoder
    memory = model.encode(src)

    bos_id = tokenizer.pad_token_id  # MarianMT mồi bằng pad_token_id
    eos_id = tokenizer.eos_token_id if tokenizer.eos_token_id is not None else 0

    candidates = [([bos_id], 0.0)]
    completed_hypotheses = []

    for step in range(max_len):
        tgt_tensor = torch.tensor([c[0] for c in candidates], device=device)
        cur_beam_size = tgt_tensor.size(0)

        memory_expanded = memory.repeat(cur_beam_size, 1, 1)
        mask_expanded = src_padding_mask.repeat(cur_beam_size, 1)

        dec_output = model.decode(tgt_tensor, memory_expanded, src_padding_mask=mask_expanded)
        logits = model.fc_out(dec_output[:, -1, :])
        log_probs = torch.log_softmax(logits, dim=-1)

        all_candidates = []
        for beam_idx, (tokens, cum_score) in enumerate(candidates):
            banned_tokens = get_banned_tokens(tokens, no_repeat_ngram_size)
            if banned_tokens:
                log_probs[beam_idx, banned_tokens] = -float("inf")

            topk_log_probs, topk_tokens = torch.topk(log_probs[beam_idx], k=beam_size)

            for lp, tok in zip(topk_log_probs.tolist(), topk_tokens.tolist()):
                all_candidates.append((tokens + [tok], cum_score + lp))

        all_candidates = sorted(all_candidates, key=lambda x: x[1], reverse=True)

        candidates = []
        for tokens, score in all_candidates:
            if tokens[-1] == eos_id:
                seq_len = len(tokens) - 1
                length_penalty = ((5.0 + seq_len) / 6.0) ** length_penalty_alpha
                completed_hypotheses.append((tokens, score / length_penalty))
            else:
                candidates.append((tokens, score))

            if len(candidates) == beam_size:
                break

        if len(completed_hypotheses) >= beam_size or len(candidates) == 0:
            break

    if completed_hypotheses:
        best_tokens, _ = max(completed_hypotheses, key=lambda x: x[1])
    else:
        best_tokens, _ = max(
            candidates,
            key=lambda x: x[1] / (((5.0 + len(x[0]) - 1) / 6.0) ** length_penalty_alpha)
        )

    return tokenizer.decode(best_tokens[1:], skip_special_tokens=True).strip()

# 5. Duyệt toàn bộ tập Test để sinh câu dịch
predictions = []
references = []

# Tự động nhận diện trường tên cột ("en"/"vi" hoặc dạng lồng "translation")
sample_item = test_dataset[0]
src_col = "en" if "en" in sample_item else "src"
tgt_col = "vi" if "vi" in sample_item else "tgt"

print("\n--- BAT DAU DICH TOAN BO TAP TEST ---")
for sample in tqdm(test_dataset, desc="Generating translations"):
    src_text = sample["translation"]["en"]
    ref_text = sample["translation"]["vi"]
    # if "translation" in sample and isinstance(sample["translation"], dict):
    #     src_text = sample["translation"]["en"]
    #     ref_text = sample["translation"]["vi"]
    # else:
    #     src_text = sample[src_col]
    #     ref_text = sample[tgt_col]

    pred_text = translate_sentence(src_text, beam_size=4, max_len=60)
    
    predictions.append(pred_text)
    references.append(ref_text)

# 6. Tính toán điểm số SacreBLEU chuẩn
bleu = sacrebleu.corpus_bleu(predictions, [references])
chrf = sacrebleu.corpus_chrf(predictions, [references])
ter = sacrebleu.corpus_ter(predictions, [references])

print("\n" + "=" * 50)
print("KET QUA DANH GIA CHINH THUC TREN TAP TEST")
print("=" * 50)
print(f"SacreBLEU Score : {bleu.score:.2f}")
print(f"BLEU Breakdown  : {bleu.prec_str} (BP = {bleu.bp:.3f}, ratio = {bleu.sys_len / bleu.ref_len:.3f})")
print(f"chrF++ Score    : {chrf.score:.2f}")
print(f"TER Score       : {ter.score:.2f}")
print("=" * 50)

# 7. Xuất 20 mẫu ngẫu nhiên và lưu file kết quả để kiểm tra định tính
OUTPUT_TXT_PATH = "/kaggle/working/test_evaluation_results.txt"
with open(OUTPUT_TXT_PATH, "w", encoding="utf-8") as f:
    f.write(f"SacreBLEU: {bleu.score:.2f} | chrF++: {chrf.score:.2f} | TER: {ter.score:.2f}\n\n")
    for i in range(min(50, len(predictions))):
        f.write(f"[{i+1}]\n")
        f.write(f"REF:  {references[i]}\n")
        f.write(f"PRED: {predictions[i]}\n\n")

print(f"\nDa luu 50 mau ket qua doi chieu chi tiet tai: {OUTPUT_TXT_PATH}")

So luong mau tap Test: 19151
Dang nap checkpoint tu: /kaggle/input/notebooks/khnhhngnguynhu/train-phomt-mt/checkpoints/bestv4.pt
Nap thanh cong! Epoch: 10 | Best Val Loss: 2.8774

--- BAT DAU DICH TOAN BO TAP TEST ---


Generating translations:   0%|          | 0/19151 [00:00<?, ?it/s]


KeyError: 'translation'

In [10]:
import os
import math
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, DataCollatorForSeq2Seq
from datasets import load_from_disk
from tqdm import tqdm
import sacrebleu

# 1. Thiet lap thiet bi va tokenizer
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer_name = 'Helsinki-NLP/opus-mt-en-vi'
tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
pad_idx = tokenizer.pad_token_id
eos_idx = tokenizer.eos_token_id if tokenizer.eos_token_id is not None else 0

# 2. Nap dataset
SAVE_DATA = "/kaggle/input/datasets/khnhhngnguynhu/dataset-phomt/datasets"
loaded_datasets = load_from_disk(SAVE_DATA)
test_dataset = loaded_datasets['test']

# Su dung DataCollator de tu dong padding dong theo batch
collate_fn = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True,
    pad_to_multiple_of=8,
    return_tensors="pt"
)

TEST_BATCH_SIZE = 64
test_dataloader = DataLoader(
    test_dataset,
    batch_size=TEST_BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)

# 3. Khoi tao model va nap checkpoint
model = Seq2SeqTransformerOptimized(
    src_vocab_size=len(tokenizer),
    tgt_vocab_size=len(tokenizer),
    d_model=512,
    nhead=8,
    num_encoder_layers=4,
    num_decoder_layers=4,
    dim_feedforward=1024,
    dropout=0.1,
    pad_idx=pad_idx
).to(device)

CHECKPOINT_PATH = "/kaggle/input/notebooks/khnhhngnguynhu/train-phomt-mt/checkpoints/bestv4.pt"
if not os.path.exists(CHECKPOINT_PATH):
    # Du phong duong dan neu chua tao thu muc checkpoints
    CHECKPOINT_PATH = "/kaggle/working/bestv4.pt"

print(f"Dang nap checkpoint tu: {CHECKPOINT_PATH}")
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print(f"Nap thanh cong! Epoch: {checkpoint.get('epoch', 'N/A')} | Val Loss: {checkpoint.get('val_loss', 0.0):.4f}")


# 4. Ham batched greedy decode de chay nhanh tren 19,151 cau
@torch.no_grad()
def batched_greedy_decode(model, src, src_padding_mask, max_len=60):
    batch_size = src.size(0)
    
    # Encode toan bo batch mot lan duy nhat
    memory = model.transformer.encoder(
        model.pos_encoder(model.src_embedding(src) * math.sqrt(model.d_model)),
        src_key_padding_mask=src_padding_mask
    )

    # Khoi tao token BOS cho tat ca cac cau trong batch
    ys = torch.full((batch_size, 1), fill_value=pad_idx, dtype=torch.long, device=device)
    finished = torch.zeros(batch_size, dtype=torch.bool, device=device)

    for _ in range(max_len):
        tgt_len = ys.size(1)
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt_len, device=device).bool()
        
        tgt_emb = model.pos_encoder(model.tgt_embedding(ys) * math.sqrt(model.d_model))
        out = model.transformer.decoder(
            tgt=tgt_emb,
            memory=memory,
            tgt_mask=tgt_mask,
            memory_key_padding_mask=src_padding_mask
        )
        
        # Lay logits cua vi tri cuoi cung
        logits = model.fc_out(out[:, -1, :])
        next_tokens = torch.argmax(logits, dim=-1)

        # Neu cau da gap EOS thi giu nguyen PAD, nguoc lai nhan next_token
        next_tokens = torch.where(finished, torch.tensor(pad_idx, device=device), next_tokens)
        ys = torch.cat([ys, next_tokens.unsqueeze(1)], dim=1)

        # Cap nhat trang thai ket thuc
        finished = finished | (next_tokens == eos_idx)
        if finished.all():
            break

    return ys


# 5. Vong lap danh gia tren toan bo test set
all_predictions = []
all_references = []

print(f"\nDang bat dau danh gia {len(test_dataset)} mau tren tap Test...")
for batch in tqdm(test_dataloader, desc="Testing SacreBLEU"):
    input_ids = batch["input_ids"].to(device, non_blocking=True)
    labels = batch["labels"].to(device, non_blocking=True)
    src_padding_mask = (input_ids == pad_idx)

    # Giai ma ca batch
    with torch.amp.autocast('cuda'):
        pred_ids = batched_greedy_decode(model, input_ids, src_padding_mask, max_len=60)

    # Chuyen labels va predictions thanh text
    decoded_preds = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    
    # Thay the -100 thanh pad_id de decode nhan chuan
    labels_cleaned = torch.where(labels != -100, labels, torch.tensor(pad_idx, device=device))
    decoded_refs = tokenizer.batch_decode(labels_cleaned, skip_special_tokens=True)

    # Loai bo khoang trang thua
    all_predictions.extend([p.strip() for p in decoded_preds])
    all_references.extend([r.strip() for r in decoded_refs])

# 6. Tinh diem SacreBLEU chuan quoc te
# SacreBLEU yeu cau references o dang danh sach cac he tham chieu: [[ref1, ref2, ...]]
bleu_score = sacrebleu.corpus_bleu(all_predictions, [all_references])

print("\n" + "=" * 50)
print(f"KET QUA TEST TREN TAP PHOMT ({len(test_dataset)} SAMPLES):")
print(f"SacreBLEU Score: {bleu_score.score:.2f}")
print(f"Chi tiet n-gram: {bleu_score.prec_html}")
print(f"Length Ratio   : {bleu_score.sys_len / bleu_score.ref_len:.4f}")
print("=" * 50)

# 7. In 5 cau mau de kiem tra chat luong thuc te
print("\n--- 5 CAU VI DU KIEM TRA THUC TE ---")
for i in range(min(5, len(all_predictions))):
    print(f"Target: {all_references[i]}")
    print(f"Pred  : {all_predictions[i]}")
    print("-" * 50)

# 8. Luu ket qua du doan ra file txt
OUTPUT_TXT = "/kaggle/working/test_predictions.txt"
with open(OUTPUT_TXT, "w", encoding="utf-8") as f:
    for pred in all_predictions:
        f.write(pred + "\n")
print(f"Da xuat toan bo ban dich tap test ra: {OUTPUT_TXT}")

Dang nap checkpoint tu: /kaggle/input/notebooks/khnhhngnguynhu/train-phomt-mt/checkpoints/bestv4.pt
Nap thanh cong! Epoch: 10 | Val Loss: 2.8774

Dang bat dau danh gia 19151 mau tren tap Test...


Testing SacreBLEU: 100%|██████████| 300/300 [01:35<00:00,  3.16it/s]



KET QUA TEST TREN TAP PHOMT (19151 SAMPLES):
SacreBLEU Score: 32.60


AttributeError: 'BLEUScore' object has no attribute 'prec_html'

In [12]:
print("=" * 60)
print(f"BÁO CÁO ĐÁNH GIÁ CHÍNH THỨC TRÊN TẬP PHOMT TEST")
print(f"Số lượng mẫu kiểm thử : {len(test_dataset):,} câu")
print(f"SacreBLEU Score        : {bleu_score.score:.2f}")
print(f"Độ chính xác n-gram    : {bleu_score.prec_str}")
print(f"Brevity Penalty (BP)   : {bleu_score.bp:.4f}")
print(f"Tỷ lệ độ dài (Sys/Ref) : {bleu_score.sys_len / bleu_score.ref_len:.4f}")
print("=" * 60)

# In 5 câu dịch mẫu đã lưu trong danh sách
print("\n--- 5 CÂU DỊCH THỰC TẾ TRÊN TẬP TEST ---")
for i in range(min(5, len(all_predictions))):
    print(f"Target : {all_references[i]}")
    print(f"Model  : {all_predictions[i]}")
    print("-" * 60)

BÁO CÁO ĐÁNH GIÁ CHÍNH THỨC TRÊN TẬP PHOMT TEST
Số lượng mẫu kiểm thử : 19,151 câu
SacreBLEU Score        : 32.60
Độ chính xác n-gram    : 62.4/40.0/27.8/20.4
Brevity Penalty (BP)   : 0.9450
Tỷ lệ độ dài (Sys/Ref) : 0.9464

--- 5 CÂU DỊCH THỰC TẾ TRÊN TẬP TEST ---
Target : Anh Albert Barnett và chị Susan Barnett, thuộc hội thánh West ở Tuscaloosa, Alabama
Model  : anh Albert QC và vợ, chị Susan QC, từ Hội thánh West ở Tuscaloosa, Alabama
------------------------------------------------------------
Target : Ngày 11 và 12-1-2020, những cơn bão lớn đã quét qua và phá huỷ nhiều vùng ở miền nam và miền trung Hoa Kỳ.
Model  : bão dữ dội đã xé toạc các phần của miền nam và Trung Tây Hoa Kỳ vào ngày 11 tháng 1 và 12 năm 2020.
------------------------------------------------------------
Target : Những trận mưa to và gió lớn trong suốt hai ngày cùng với nhiều cơn lốc xoáy đã gây thiệt hại nặng nề cho nhiều bang.
Model  : hai ngày mưa lớn, gió mạnh và nhiều cơn lốc gây thiệt hại lớn ở nhiều bang.

In [13]:
import math
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, DataCollatorForSeq2Seq
from datasets import load_from_disk
from tqdm import tqdm
import sacrebleu

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer_name = 'Helsinki-NLP/opus-mt-en-vi'
tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
pad_idx = tokenizer.pad_token_id
eos_idx = tokenizer.eos_token_id if tokenizer.eos_token_id is not None else 0

# 1. Ham Batched Beam Search vector hoa hoan toan tren GPU
@torch.no_grad()
def batched_beam_search(
    model: nn.Module,
    src: torch.Tensor,
    src_padding_mask: torch.Tensor,
    beam_size: int = 4,
    max_len: int = 60,
    alpha: float = 0.8  # alpha > 0.6 tang trong so do dai, keo Length Ratio len ~1.0
) -> torch.Tensor:
    batch_size = src.size(0)

    # 1. Encode 1 lan duy nhat cho toan bo batch
    src_emb = model.pos_encoder(model.src_embedding(src) * math.sqrt(model.d_model))
    memory = model.transformer.encoder(src_emb, src_key_padding_mask=src_padding_mask)

    # 2. Mo rong memory va mask theo beam_size: (B * beam_size, S, D)
    memory_expanded = memory.repeat_interleave(beam_size, dim=0)
    src_mask_expanded = src_padding_mask.repeat_interleave(beam_size, dim=0)

    # Khoi tao token BOS cho moi beam: (B, beam_size, 1)
    ys = torch.full((batch_size, beam_size, 1), fill_value=pad_idx, dtype=torch.long, device=device)

    # Bang diem tich luy: (B, beam_size)
    # Khoi tao beam 0 = 0.0, cac beam con lai = -inf de tranh trung lap o buoc 1
    beam_scores = torch.full((batch_size, beam_size), fill_value=-1e9, device=device)
    beam_scores[:, 0] = 0.0

    # Theo doi trang thai da gap EOS: (B, beam_size)
    done_mask = torch.zeros((batch_size, beam_size), dtype=torch.bool, device=device)

    for step in range(max_len):
        current_len = ys.size(2)
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(current_len, device=device).bool()

        # Reshape de decode song song tren toan bo batch * beam: (B * beam_size, current_len)
        flat_ys = ys.view(batch_size * beam_size, current_len)

        tgt_emb = model.pos_encoder(model.tgt_embedding(flat_ys) * math.sqrt(model.d_model))
        dec_out = model.transformer.decoder(
            tgt=tgt_emb,
            memory=memory_expanded,
            tgt_mask=tgt_mask,
            memory_key_padding_mask=src_mask_expanded
        )

        # Lay logits tai vi tri token cuoi cung va tinh log-softmax
        logits = model.fc_out(dec_out[:, -1, :])  # (B * beam_size, vocab_size)
        log_probs = torch.log_softmax(logits, dim=-1).view(batch_size, beam_size, -1)

        # Neu mot nhanh da gap EOS, chi cho phep sinh tiep PAD voi xac suat 1.0 (log_prob = 0)
        eos_penalty = torch.full_like(log_probs, fill_value=-1e9)
        eos_penalty[:, :, pad_idx] = 0.0
        log_probs = torch.where(done_mask.unsqueeze(-1), eos_penalty, log_probs)

        # Cong don diem xac suat: (B, beam_size, vocab_size)
        candidate_scores = beam_scores.unsqueeze(-1) + log_probs
        vocab_size = log_probs.size(-1)
        candidate_scores = candidate_scores.view(batch_size, -1)  # (B, beam_size * vocab_size)

        # Chon top 2 * beam_size ung vien tot nhat
        topk_scores, topk_indices = torch.topk(candidate_scores, k=beam_size, dim=-1)

        # Giai ma chi so beam cha va token moi
        parent_beams = topk_indices // vocab_size   # (B, beam_size)
        next_tokens = topk_indices % vocab_size     # (B, beam_size)

        # Thu thap chuoi token tu beam cha
        batch_indices = torch.arange(batch_size, device=device).unsqueeze(1).expand_as(parent_beams)
        selected_ys = ys[batch_indices, parent_beams]  # (B, beam_size, current_len)
        ys = torch.cat([selected_ys, next_tokens.unsqueeze(-1)], dim=-1)

        # Cap nhat trang thai done_mask va beam_scores
        parent_done = done_mask[batch_indices, parent_beams]
        new_done = (next_tokens == eos_idx) | (next_tokens == pad_idx)
        done_mask = parent_done | new_done
        beam_scores = topk_scores

        # Neu tat ca cac beam cua tat ca batch deu hoan thanh thi dung som
        if done_mask.all():
            break

    # 3. Chuan hoa do dai (Length Penalty) de chon chuoi tot nhat moi batch
    # Cong thuc Google NMT: LP = ((5 + L) / 6) ^ alpha
    seq_lengths = (ys != pad_idx).sum(dim=-1).float()
    lp = ((5.0 + seq_lengths) / 6.0) ** alpha
    normalized_scores = beam_scores / lp

    best_beam_indices = torch.argmax(normalized_scores, dim=1)  # (B,)
    best_sequences = ys[torch.arange(batch_size, device=device), best_beam_indices]  # (B, total_len)

    return best_sequences


# 2. Nap dataset va Checkpoint
SAVE_DATA = "/kaggle/input/datasets/khnhhngnguynhu/dataset-phomt/datasets"
loaded_datasets = load_from_disk(SAVE_DATA)
test_dataset = loaded_datasets['test']

collate_fn = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True,
    pad_to_multiple_of=8,
    return_tensors="pt"
)

# Chay batch_size=32 khi beam_size=4 de toi uu VRAM (~6-8GB)
TEST_BATCH_SIZE = 32
test_dataloader = DataLoader(
    test_dataset,
    batch_size=TEST_BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)

CHECKPOINT_PATH = "/kaggle/input/notebooks/khnhhngnguynhu/train-phomt-mt/checkpoints/bestv4.pt"
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

# 3. Vong lap danh gia tren toan bo tap Test
all_predictions = []
all_references = []

print(f"Bat dau danh gia Batched Beam Search tren {len(test_dataset):,} mau...")
for batch in tqdm(test_dataloader, desc="Batched Beam Search"):
    input_ids = batch["input_ids"].to(device, non_blocking=True)
    labels = batch["labels"].to(device, non_blocking=True)
    src_padding_mask = (input_ids == pad_idx)

    with torch.amp.autocast('cuda'):
        pred_ids = batched_beam_search(
            model=model,
            src=input_ids,
            src_padding_mask=src_padding_mask,
            beam_size=4,
            max_len=60,
            alpha=0.8
        )

    # Giai ma chuoi token
    decoded_preds = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    labels_cleaned = torch.where(labels != -100, labels, torch.tensor(pad_idx, device=device))
    decoded_refs = tokenizer.batch_decode(labels_cleaned, skip_special_tokens=True)

    all_predictions.extend([p.strip() for p in decoded_preds])
    all_references.extend([r.strip() for r in decoded_refs])

# 4. Tinh SacreBLEU
bleu_score = sacrebleu.corpus_bleu(all_predictions, [all_references])

print("\n" + "=" * 60)
print("KET QUA SO SANH VOI GREEDY SEARCH:")
print(f"SacreBLEU Score        : {bleu_score.score:.2f} (Truoc: 32.60)")
print(f"Do chinh xac n-gram    : {bleu_score.prec_str}")
print(f"Brevity Penalty (BP)   : {bleu_score.bp:.4f} (Truoc: 0.9450)")
print(f"Ty le do dai (Sys/Ref) : {bleu_score.sys_len / bleu_score.ref_len:.4f} (Truoc: 0.9464)")
print("=" * 60)

Bat dau danh gia Batched Beam Search tren 19,151 mau...


Batched Beam Search: 100%|██████████| 599/599 [05:21<00:00,  1.86it/s]



KET QUA SO SANH VOI GREEDY SEARCH:
SacreBLEU Score        : 34.22 (Truoc: 32.60)
Do chinh xac n-gram    : 63.4/41.6/29.4/21.8
Brevity Penalty (BP)   : 0.9495 (Truoc: 0.9450)
Ty le do dai (Sys/Ref) : 0.9507 (Truoc: 0.9464)


In [3]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input/models/khnhhngnguynhu/phomt-mariannmt/pytorch/epoch1-2/2/bestv3'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# # Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# # Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

# import kagglehub
# # kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
import os
import shutil
import torch

# 1. Thư mục chứa các file unzipped trên Kaggle Input
EXTRACTED_DIR = "/kaggle/input/notebooks/khnhhngnguynhu/train-phomt-mt/checkpoints/bestv4.pt"

# 2. Đóng gói lại thành 1 file zip/pt chuẩn trong thư mục working
os.makedirs("/kaggle/working/haha")
REPACKED_PT_PATH = "/kaggle/working/haha/bestv3.pt"

if not os.path.exists(REPACKED_PT_PATH):
    # Nén toàn bộ nội dung bên trong thư mục EXTRACTED_DIR thành file zip
    archive_path = shutil.make_archive(
        base_name="/kaggle/working/restored_bestv3", 
        format="zip", 
        root_dir=EXTRACTED_DIR
    )
    # Đổi tên file .zip thành .pt
    os.rename(archive_path, REPACKED_PT_PATH)
    print(f"--> Đã đóng gói lại checkpoint thành công tại: {REPACKED_PT_PATH}")

# # 3. Nạp bình thường vào model
# checkpoint = torch.load(REPACKED_PT_PATH, map_location=device)
# model.load_state_dict(checkpoint["model_state_dict"])
# print(f"Nạp thành công checkpoint từ Epoch {checkpoint.get('epoch')} (Val Loss: {checkpoint.get('val_loss'):.4f})")

NotADirectoryError: [Errno 20] Not a directory: '/kaggle/input/notebooks/khnhhngnguynhu/train-phomt-mt/checkpoints/bestv4.pt'

In [ ]:
import os
from IPython.display import FileLink

# 1. Sao chép file từ Input sang Output (Working)
!cp /kaggle/input/notebooks/khnhhngnguynhu/train-phomt-mariannmt/checkpoints/vienv2.pt /kaggle/working/vienv2.pt

# 2. Tạo liên kết tải xuống từ thư mục Output
FileLink(r'bestv4.pt')
